In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

# Q1

In [5]:
train.iloc[150]["answer"]
# Therefore according to label it is 2.

'C'

# Q2

In [6]:
row0 = train.iloc[0]
prompt = row0["prompt"]
optB = row0["B"]
formatted_optB = str(prompt)+" [SEP] "+str(optB)
len(formatted_optB)

407

# Q3

In [7]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
import torch
options = ["A","B","C","D","E"]
input_ids = []
for i in options:
    prompt = row0["prompt"]
    opt = row0[i]
    formatted_opt = str(prompt)+" [SEP] "+str(opt)
    enc = tokenizer(formatted_opt,padding="max_length",truncation=True,max_length=128,return_tensors="pt")
    input_ids.append(enc['input_ids'])
inputs = torch.stack(input_ids)    

In [9]:
reshaped = inputs.permute(1,0,2)
reshaped[:, 0, :]

tensor([[  101,  4060,  1996,  2190,  2825,  3437,  1024,  2054,  2003,  3235,
          2002,  5178, 13327,  1005,  1055,  3193,  2006,  1996,  3276,  2090,
          2051,  1998,  2529,  4598,  1029,  2426,  1996,  3205,  7047,  1012,
           102,  3235,  2002,  5178, 13327,  7164,  2008,  4286,  4839,  2306,
          1037,  2051, 22961,  2008,  2003, 10709,  1998,  2515,  2025,  2031,
          1037,  4225,  2927,  2030,  2203,  1012,  1996,  3276,  2000,  1996,
          2627,  7336, 21894,  2009,  2004,  1037,  3439,  3690,  1010,  1998,
          1996,  3276,  2000,  1996,  2925,  7336,  4526,  1037,  2088,  2008,
          2097, 18094,  3458,  2028,  1005,  1055,  2219,  2051,  1012,   102,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,  

# Q4

In [10]:
first_16 = train.head(16)
prompts = first_16['prompt'].tolist()

flat_formatted_inputs = []
for idx,row in first_16.iterrows():
    prompt = row['prompt']
    for opt in options:
        formatted = str(prompt)+" [SEP] "+str(row[opt])
        flat_formatted_inputs.append(formatted)

encoded_outputs = tokenizer(
    flat_formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

final_input_ids = encoded_outputs['input_ids'].view(16,5,128)

print(f"Total token positions are: {final_input_ids.numel()}")

Total token positions are: 10240


# Q5

In [11]:
from transformers import AutoModelForMultipleChoice
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
print(f"Total Number of Trainable Params: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total Number of Trainable Params: 109483009


In [12]:
model.eval()
options = ['A','B','C','D','E']
row0 = train.iloc[0]
formatted_opts = [str(row0["prompt"])+" [SEP] "+str(row0[opt]) for opt in options]

encoded_inputs = tokenizer(
    formatted_opts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

model_inputs = {
    "input_ids": encoded_inputs["input_ids"].view(1, 5, 128),
    "attention_mask": encoded_inputs["attention_mask"].view(1, 5, 128)
}

with torch.no_grad():
    outputs = model(**model_inputs)

logits = outputs.logits
print(f"Logits shape: {logits.shape}")

Logits shape: torch.Size([1, 5])


# Q6

In [13]:
correct_label = torch.tensor([1])
model_inputs = {
    "input_ids": encoded_inputs["input_ids"].view(1, 5, 128),
    "attention_mask": encoded_inputs["attention_mask"].view(1, 5, 128),
    "labels": correct_label
}

with torch.no_grad():
    outputs = model(**model_inputs)

loss = outputs.loss
print(f"Loss: {loss}")
print(f"Loss Dimension: {loss.shape}")

Loss: 1.6029059886932373
Loss Dimension: torch.Size([])


# Q7

In [4]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.8 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [15]:
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

lora_model = get_peft_model(model,lora_config)

print(f"Total Number of Trainable Params: {sum(p.numel() for p in lora_model.parameters() if p.requires_grad)}")
print(f"Total Number of Trainable Params: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Total Number of Trainable Params: 295681
Total Number of Trainable Params: 295681


# Q8

In [16]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

label_mapping = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

df_100 = train.head(100)
raw_dataset = Dataset.from_pandas(df_100)

def preprocess_mcq(examples):
    options_keys = ["A", "B", "C", "D", "E"]
    batch_input_ids = []
    batch_attention_mask = []
    for i in range(len(examples["prompt"])):
        prompt = examples["prompt"][i]
        formatted_opts = [str(prompt) + " [SEP] " + str(examples[opt][i]) for opt in options_keys]
        tokenized = tokenizer(
            formatted_opts,
            padding="max_length",
            truncation=True,
            max_length=128
        )
        batch_input_ids.append(tokenized["input_ids"])
        batch_attention_mask.append(tokenized["attention_mask"])
    encoded_labels = [label_mapping[label] for label in examples["answer"]]
    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels": encoded_labels
    }
processed_dataset = raw_dataset.map(
    preprocess_mcq, 
    batched=True,
    remove_columns=raw_dataset.column_names
)
first_item = processed_dataset[0]
print("First item input_ids shape:", torch.tensor(first_item["input_ids"]).shape)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

First item input_ids shape: torch.Size([5, 128])


# Q9

In [5]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType

train_subset = train.head(32)

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForMultipleChoice.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
peft_model = get_peft_model(base_model, lora_config)

raw_dataset = Dataset.from_pandas(train_subset)

label_mapping = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
options_keys = ["A", "B", "C", "D", "E"]

def preprocess_function(examples):
    batch_input_ids = []
    batch_attention_mask = []
    
    for i in range(len(examples["prompt"])):
        prompt = examples["prompt"][i]
        formatted_opts = [str(prompt) + " [SEP] " + str(examples[opt][i]) for opt in options_keys]
        
        tokenized = tokenizer(
            formatted_opts,
            padding="max_length",
            truncation=True,
            max_length=64 
        )
        batch_input_ids.append(tokenized["input_ids"])
        batch_attention_mask.append(tokenized["attention_mask"])
        
    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels": [label_mapping[ans] for ans in examples["answer"]]
    }

processed_dataset = raw_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=raw_dataset.column_names
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=5e-4,
    per_device_train_batch_size=4,       
    gradient_accumulation_steps=1,       
    max_steps=4,                         
    logging_steps=1,
    report_to="none",
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=processed_dataset
)

train_output = trainer.train()

print(f"Global Step: {train_output.global_step}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

... (2 outputs left out)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
1,3.362556
2,3.204622
3,3.394570
4,3.371872


Global Step: 4


# Q10

In [6]:
import torch.nn.functional as F

first_item = processed_dataset[0]

model_inputs = {
    "input_ids": torch.tensor([first_item['input_ids']]).to(peft_model.device),
    "attention_mask": torch.tensor([first_item['attention_mask']]).to(peft_model.device)
}

peft_model.eval()
with torch.no_grad():
    output = peft_model(**model_inputs)
    
logits = output.logits
probs = F.softmax(logits,dim=-1)
probs

tensor([[0.1854, 0.1862, 0.1940, 0.2152, 0.2192]], device='cuda:0')

In [17]:
train.iloc[0]['answer']

'B'

In [7]:
peft_model.print_trainable_parameters()

for name, param in peft_model.named_parameters():
    if param.requires_grad:
        print(name)

trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693
base_model.model.bert.encoder.layer.0.attention.self.query.lora_A.default.weight
base_model.model.bert.encoder.layer.0.attention.self.query.lora_B.default.weight
base_model.model.bert.encoder.layer.0.attention.self.value.lora_A.default.weight
base_model.model.bert.encoder.layer.0.attention.self.value.lora_B.default.weight
base_model.model.bert.encoder.layer.1.attention.self.query.lora_A.default.weight
base_model.model.bert.encoder.layer.1.attention.self.query.lora_B.default.weight
base_model.model.bert.encoder.layer.1.attention.self.value.lora_A.default.weight
base_model.model.bert.encoder.layer.1.attention.self.value.lora_B.default.weight
base_model.model.bert.encoder.layer.2.attention.self.query.lora_A.default.weight
base_model.model.bert.encoder.layer.2.attention.self.query.lora_B.default.weight
base_model.model.bert.encoder.layer.2.attention.self.value.lora_A.default.weight
base_model.model.bert.encoder.laye